# 📗 서빙 성능 벤치마킹: vLLM 노무 상담 서버의 지연, 처리량, 비용 측정하기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<div style="background:#ffffff;color:#202124;font-size:17px;line-height:1.6;padding:6px 10px;border-left:4px solid #54769b;">
<strong style="color:#1f3a5f;font-size:19px;">실습 환경</strong><br>
<strong style="font-size:1.13em;color:#1f3a5f;">실행:</strong> RunPod의 Linux + NVIDIA GPU에서 vLLM 0.27.0 서버, 벤치마크, 노트북을 모두 실행합니다. <strong style="font-size:1.13em;color:#1f3a5f;">내 PC에서 실행하는 코드는 없습니다.</strong> VS Code 창은 내 PC에 열리지만, 그 안의 노트북과 터미널은 RunPod에서 실행됩니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">접속:</strong> VS Code Remote SSH로 파드에 접속해 원격 노트북과 터미널을 사용합니다. API 요청은 같은 파드의 127.0.0.1:8000으로 보냅니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">준비:</strong> 딥러닝 학습 원리 단원 <a style="color:#245fa0;text-decoration:underline;" href="../day65_딥러닝_학습원리/부록_RunPod_GPU에서_실습하기.ipynb">RunPod 부록 5~7절</a>에 따라 원격 저장소를 열고, 저장소의 .venv 커널을 선택합니다.<br>
<strong style="font-size:1.13em;color:#1f3a5f;">최소 권장:</strong> GPU VRAM 24GB 1장 · RAM 16GB · 디스크 80GB. 서버는 하나만 실행합니다.
</div>
</div>

#### **서빙 성능 벤치마킹**

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서빙 중인 LLM 서버에 <strong style="font-size:1.13em;color:#1f3a5f;">실제 서비스와 같은 요청을 정한 동시 요청 수로 보내, 사용자가 기다리는 지연과 서버가 만드는 처리량을 계산하는 일</strong>입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배우는 이유:</strong> 서버 한 대에 동시 요청을 몇 개까지 보낼지, 출력 토큰 100만 개에 비용이 얼마인지를 측정값으로 정합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">활용처:</strong> 상담 봇·RAG 서비스의 서버 대수와 설정을 정할 때, 모델이나 서버 옵션을 바꾼 뒤 지연과 처리량을 비교할 때 씁니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/05_serving_benchmark.png" width="1050" alt="질문 여러 개를 GPU 서버에 동시에 보내고, 첫 토큰까지의 기다림, 1초에 만드는 토큰 수(처리량), 100만 토큰당 비용을 숫자로 비교합니다.">

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">근로기준법 조문을 참고하는 노무 상담 서버를 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod에서 vLLM으로 띄우고</strong>, <strong style="font-size:1.13em;color:#1f3a5f;">같은 파드의 원격 노트북에서 요청 하나의 지연</strong>을 계산합니다. 이어서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench serve</code>로 부하를 걸고, 저장된 결과로 지연 목표 안의 동시 요청 수와 100만 토큰당 비용까지 계산합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/00_lesson01_goals.png" width="1050" alt="학습 목표 그림: 1 요청 하나의 지표, 2 vllm bench serve 부하 측정, 3 동시 요청 수와 지연 목표, 4 100만 토큰당 비용">

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

**학습 목표** (그림의 번호와 같습니다)

1. 스트리밍 응답 하나로 TTFT, TPOT, ITL, 요청 지연을 계산합니다.
2. `vllm bench serve`로 동시 요청 부하를 걸고, RunPod 화면의 요약 테이블에서 평균과 p50, p95, p99, 출력 처리량을 읽습니다.
3. 동시 요청 수를 늘리며 지연과 출력 처리량을 비교하고, 지연 목표를 지키면서 출력 처리량이 가장 큰 동시 요청 수를 고릅니다.
4. 출력 처리량과 GPU 시간당 요금으로 100만 토큰당 비용을 계산합니다.

</div>

## 1. 실습 준비와 서버 실행

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버는 모델을 내려받아 GPU에 올리는 데 몇 분이 걸립니다. 그래서 <strong style="font-size:1.13em;color:#1f3a5f;">키 생성과 서버 실행을 먼저 해 두고</strong>, 서버가 뜨는 동안 노트북을 준비합니다.</p>
</div>

### 1-1. 원격 노트북과 터미널의 역할

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VS Code Remote SSH로 RunPod에 접속한 원격 창에서 실습합니다. 원격 실습 저장소(단원 폴더와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pyproject.toml</code>이 있는 폴더)를 열고, 노트북 오른쪽 위 <strong style="font-size:1.13em;color:#1f3a5f;">Select Kernel → Python Environments... → .venv</strong>를 선택합니다. 접속과 환경 준비는 앞에서 연결한 RunPod 부록 5~7절을 따릅니다.</p>
<p style="margin:6px 0;">RunPod에서 빌린 GPU 컴퓨터 한 대를 <strong style="font-size:1.13em;color:#1f3a5f;">파드(Pod)</strong>라고 부릅니다. 아래 노트북과 터미널은 모두 같은 파드에서 실행하고 같은 단원 폴더의 파일을 씁니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">작업</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">실행할 곳</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">역할</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm serve</code> 실행과 종료(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>), 종료 뒤 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">원격 터미널 1</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">모델을 GPU에 올리고 요청을 받습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench serve</code>, 시작 로그 확인</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">원격 터미널 2</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">부하를 걸고 서버 상태를 확인합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">처리량 로그 실시간 보기(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tail -f</code>)</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">원격 터미널 3</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">벤치마크 도중 서버 상태를 봅니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">요청 하나 스트리밍, 입력 파일 생성, 결과 JSON·그래프·비용 분석</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">원격 노트북</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">같은 파드의 API를 호출하고 측정값을 분석합니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">아래 Python 셀은 모두 원격 노트북에서 실행합니다.</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">RunPod 터미널</code> 라벨이 붙은 명령은 같은 원격 창의 <strong style="font-size:1.13em;color:#1f3a5f;">Terminal → New Terminal</strong>에서 실행합니다. 새 터미널은 원격 저장소 폴더에서 열고, 준비 블록으로 단원 폴더에 이동합니다.</p>
</div>

### 1-2. 서버 인증 키 생성

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">먼저 원격 터미널 1에서 서버 인증 키를 생성합니다.</strong> 아래 명령은 지난 단원 부록처럼 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">enkoa-vllm-</code> 뒤에 무작위 문자열을 붙여 키를 만들고, 원격 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>로 자동 저장합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 1**

```bash
# VS Code에서 연 원격 저장소(실습자료) 폴더의 터미널에서 실행합니다
cd day76_추론가속_서빙벤치마킹
pwd   # 끝이 day76_추론가속_서빙벤치마킹이면 맞습니다
mkdir -p output/bench output/sweep
# 지난 단원 부록과 같은 방식으로 서버 인증 키를 생성합니다
VLLM_API_KEY="$(uv run python -c 'import secrets; print("enkoa-vllm-" + secrets.token_urlsafe(32))')" &&
export VLLM_API_KEY OPENAI_API_KEY="$VLLM_API_KEY" &&
uv run python -c 'import os; from dotenv import set_key; set_key("../.env", "VLLM_API_KEY", os.environ["VLLM_API_KEY"])' &&
printf 'VLLM_API_KEY=%s\n' "$VLLM_API_KEY"
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">명령어 설정</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">secrets.token_urlsafe(32)</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">무작위 문자열을 만들어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">enkoa-vllm-</code> 뒤에 붙입니다. 지난 단원 부록과 같은 키 생성 방식입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">export VLLM_API_KEY OPENAI_API_KEY=&quot;$VLLM_API_KEY&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">이 터미널의 두 환경변수에 같은 키를 넣습니다. 전자는 서버 인증에, 후자는 벤치마크 도구가 요청에 키를 붙이는 데 씁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">set_key(&quot;../.env&quot;, &quot;VLLM_API_KEY&quot;, ...)</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">원격 저장소(실습자료) 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 키를 저장합니다. 노트북과 새 터미널이 이 값을 읽습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">printf 'VLLM_API_KEY=%s\n' ...</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">생성한 키를 화면에 보여 줍니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">터미널 1은 닫지 않고 바로 아래 1-3의 서버 실행에 이어 씁니다. 키를 다시 생성했다면 서버도 새 키로 다시 시작하고 노트북 준비 셀부터 다시 실행합니다.</p>
</div>

### 1-3. RunPod에서 서버 실행

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버 모델은 지난 단원 교안 02의 양자화 비교에서 원본(bf16)으로 띄운 Hugging Face 허브 모델 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Qwen/Qwen3.5-4B</code>입니다.</p>
<p style="margin:6px 0;">키를 생성한 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>에서 바로 아래 서버 명령을 실행합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--api-key &quot;$VLLM_API_KEY&quot;</code>가 생성한 키를 서버 인증에 사용합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 1**

```bash
uv run vllm serve Qwen/Qwen3.5-4B \
  --served-model-name qwen3.5-4b \
  --max-model-len 8192 \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --enable-prompt-tokens-details \
  --enable-prefix-caching \
  --host 127.0.0.1 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/qwen3.5-4b_server.log
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">명령어 설정</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv run vllm serve Qwen/Qwen3.5-4B</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">지난 단원 교안 02에서 띄운 원본 모델 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Qwen/Qwen3.5-4B</code>으로 서버를 시작합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-model-len 8192</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">시스템 메시지·질문·답변을 합친 최대 토큰 수입니다. 참고 조문이 든 시스템 메시지가 길어 지난 단원 교안 01의 4096보다 크게 둡니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--language-model-only</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">이미지·영상 입력을 모두 끄고 텍스트 요청만 받습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--host 127.0.0.1</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">같은 파드의 노트북과 벤치마크에서만 API를 호출합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prompt-tokens-details</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">응답의 토큰 사용량에 입력 토큰 가운데 캐시에서 다시 쓴 토큰 수(캐시 적중 토큰)를 넣습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">같은 시스템 메시지처럼 요청마다 같은 앞부분의 계산 결과를 남겨 두고 다음 요청에서 다시 써, 첫 토큰이 더 빨리 나오고 GPU 시간을 아낍니다. vLLM 0.27.0은 Qwen3.5처럼 두 종류의 층을 섞어 쓰는 하이브리드 모델에서 이 기능을 기본으로 끄므로 직접 켭니다(자세한 내용은 교안 02 2절).</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">나머지 옵션(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--served-model-name</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reasoning-parser</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--default-chat-template-kwargs</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--port</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--api-key</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code>)과 메모리 예산(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--gpu-memory-utilization</code> 생략, 기본 0.92)은 지난 단원 교안 01 3-1과 같습니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 요청을 보낼 수 있습니다. 서버 출력은 RunPod의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/qwen3.5-4b_server.log</code>에도 남습니다. <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>은 그대로 두고 <strong style="font-size:1.13em;color:#1f3a5f;">같은 VS Code 원격 창의 노트북</strong>으로 돌아옵니다.</p>
<p style="margin:6px 0;">첫 실행은 모델 다운로드와 GPU 계산 준비에 시간이 걸릴 수 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버가 뜨는 동안 1-4의 노트북 준비를 진행합니다.</strong> 첫 실행이면 모델을 내려받느라 몇 분 걸립니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 서버는 끄지 않고 과제 01과 교안 02의 캐시 적중 확인까지 이어 씁니다.</strong> 같은 설정의 서버를 다시 시작하는 시간을 줄이기 위해서입니다.</p>
<p style="margin:6px 0;">서버 로그 첫머리의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">non-default args</code> 줄에 API 키가 찍힙니다. 서버 로그나 터미널 1 화면은 다른 사람에게 공유하지 않습니다.</p>
</div>

### 1-4. 노트북 준비와 상담 데이터

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">준비 셀은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 인증 키, 자료 경로, 설정값, 그래프 글꼴을 준비합니다. 서버 주소는 같은 파드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">http://127.0.0.1:8000</code>입니다. 다음 셀은 API 키를 화면에 표시하므로 실행 결과가 남은 노트북은 공유하지 않습니다.</p>
</div>

**셀 실행 · RunPod 원격 노트북** (VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다)

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from transformers import AutoTokenizer

material_dir = Path(".").resolve()  # 이 단원 폴더(정답 노트북은 "..")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
bench_dir = output_dir / "bench"           # 같은 파드에서 벤치마크가 저장한 결과 JSON
sweep_dir = output_dir / "sweep"           # 같은 파드에서 sweep이 저장한 summary.csv
bench_dir.mkdir(parents=True, exist_ok=True)
sweep_dir.mkdir(parents=True, exist_ok=True)
# 터미널에서 생성해 원격 저장소의 .env에 저장한 키를 읽습니다
load_dotenv(material_dir.parent / ".env", override=True)
SERVER_URL = "http://127.0.0.1:8000"  # 노트북과 서버가 같은 RunPod에 있습니다
VLLM_API_KEY = os.environ["VLLM_API_KEY"]

# 설정값
MODEL_NAME = "Qwen/Qwen3.5-4B"            # 서버가 불러온 허브 모델(토크나이저도 이 이름으로 받습니다)
SERVED_NAME = "qwen3.5-4b"               # 서버 이름(--served-model-name). 요청의 model에 넣습니다
MAX_MODEL_LEN = 8192                  # 참고 조문이 든 시스템 메시지 + 질문 + 답을 담는 최대 길이
MAX_TOKENS = 256                     # 요청마다 생성 상한(지난 단원과 같은 값)
TTFT_TARGET_MS = 1000                # 지연 목표: 첫 토큰까지 1000ms 안
TPOT_TARGET_MS = 50                  # 지연 목표: 출력 토큰 하나에 50ms 안

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
import platform

import matplotlib.pyplot as plt
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스: 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# 같은 파드의 서버 주소와 원격 저장소 .env에서 읽은 API 키
print("서버 주소:", SERVER_URL)
print("VLLM_API_KEY:", VLLM_API_KEY)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노무 상담 봇의 근로시간·휴가 창구를 서비스한다고 봅니다. 시스템 메시지는 지침 문장 뒤에 근로기준법 제4장(근로시간과 휴식) 조문을 붙인 것이라 모든 요청에서 같고, user 메시지에는 질문이 들어갑니다. 참고 조문은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/본실습/labor_reference.json</code>, 질문은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/본실습/labor_questions.jsonl</code>에 있습니다.</p>
</div>

In [ ]:
# 지난 단원과 같은 지침 문장. 시스템 메시지는 이 문장 뒤에 참고 조문을 붙여 만듭니다
SYSTEM_PROMPT = "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다."

def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 단원과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
# 노무 상담 창구의 참고 조문(제목과 조문 리스트)과 질문
with open(data_dir / "본실습" / "labor_reference.json", encoding="utf-8") as f:
    labor_reference = json.load(f)
labor_questions = load_jsonl(data_dir / "본실습" / "labor_questions.jsonl")
print("참고 조문:", labor_reference["title"], "| 조문 수:", len(labor_reference["articles"]), "| 질문 수:", len(labor_questions))

# 시스템 메시지 = 지침 문장 + 빈 줄 + [참고 조문] + 조문들(한 줄에 조문 하나)
labor_articles = [article["text"] for article in labor_reference["articles"]]
system_prompt = SYSTEM_PROMPT + "\n\n[참고 조문]\n" + "\n".join(labor_articles)

In [ ]:
# 서버와 같은 모델의 토크나이저로 시스템 메시지의 토큰 수를 셉니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("시스템 메시지 토큰 수:", len(tokenizer.encode(system_prompt)))

In [ ]:
# 질문 앞 3개(id, 질문, 출처 주제)
labor_question_table = pd.DataFrame(labor_questions)
display(labor_question_table.head(3))

### 1-5. 서버 시작 로그 확인

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 1에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Application startup complete</code>가 나온 뒤 진행합니다. 터미널 1은 서버가 쓰고 있으므로, 로그 확인과 벤치마크는 원격 창에서 새로 연 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 2</strong>에서 실행합니다. 원격 저장소 폴더에서 아래 준비 블록을 실행합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 2**

```bash
# 새 원격 터미널이면 이 블록부터 실행합니다
# VS Code에서 연 원격 저장소(실습자료) 폴더의 터미널에서 실행합니다
cd day76_추론가속_서빙벤치마킹
pwd   # 끝이 day76_추론가속_서빙벤치마킹이면 맞습니다
mkdir -p output/bench output/sweep
# 같은 원격 저장소의 .env에서 이미 생성한 키를 불러옵니다
VLLM_API_KEY="$(uv run python -c 'from dotenv import dotenv_values; print(dotenv_values("../.env")["VLLM_API_KEY"])')" &&
export VLLM_API_KEY OPENAI_API_KEY="$VLLM_API_KEY"
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">명령어 설정</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">dotenv_values(&quot;../.env&quot;)[&quot;VLLM_API_KEY&quot;]</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">터미널 1에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 저장한 키를 읽습니다. 키를 새로 생성하지 않습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">export VLLM_API_KEY OPENAI_API_KEY=&quot;$VLLM_API_KEY&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">터미널 1과 같은 키를 이 터미널의 두 환경변수에 넣습니다.</td>
</tr>
</tbody>
</table>
</div>

**RunPod 터미널 2 · 화면에서 확인**

```bash
grep -E "GPU KV cache size|Maximum concurrency" output/qwen3.5-4b_server.log
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">명령어 설정:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">grep -E &quot;A|B&quot;</code>는 서버 로그에서 A나 B가 든 줄만 골라 보여 줍니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong> (지난 단원에서 본 두 줄입니다)</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">GPU KV cache size</code>(KV 캐시 토큰):</strong> 이 서버가 KV 캐시에 담을 수 있는 토큰의 총량입니다. GPU 메모리 예산(기본 92%)에서 모델 가중치와 실행 공간을 뺀 나머지로 정해지므로, 같은 GPU라도 서버 설정이 바뀌면 달라집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Maximum concurrency</code>(동시 요청 추정치):</strong> 모든 요청이 최대 길이(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-model-len</code> 8,192토큰)를 꽉 채운다고 가정할 때, KV 캐시에 동시에 담을 수 있는 요청 수입니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">예: L4 GPU에서 KV 캐시 토큰이 247,994개이면 247,994 ÷ 8,192 ≈ 30.27이라 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">30.27x</code>로 찍힙니다.</li>
<li style="margin:3px 0;">가장 긴 요청만 온다고 본 보수적인 값이라, 서버가 실제로 동시에 처리하는 요청 수와 다릅니다.</li>
</ul>
</li>
</ul>
<p style="margin:6px 0;">KV 캐시는 일정한 크기의 칸(블록)으로 나눠 씁니다(지난 단원 교안 01 5-3). vLLM은 동시 요청 추정치를 먼저 계산하고, 그 값에 최대 길이를 곱해 KV 캐시 토큰으로 보여 줍니다.</p>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">동시 요청 추정치</strong> = KV 캐시 블록 수 ÷ 최대 길이 요청 하나가 쓰는 블록 수<br><strong style="color:#1f3a5f;">KV 캐시 토큰</strong> = 동시 요청 추정치 × 8,192</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실제 동시 처리 수:</strong> 실제 요청은 입력 약 5천 토큰과 답 256토큰 이하라 최대 길이보다 짧습니다. 또 시스템 메시지가 같으면 그 블록을 여러 요청이 함께 씁니다(교안 02 2절의 접두어 캐싱). 그래서 추정치보다 많은 요청이 들어갑니다. 대신 서버가 한 번에 돌리는 요청 수의 상한(이 GPU 기본 256개)과 벤치마크의 동시 요청 수를 넘지는 못합니다. 실제 값은 5-1에서 보는 처리량 로그의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Running</code>입니다.</li>
</ul>
</div>

## 2. 서빙 성능 지표

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">사용자는 답을 기다리는 시간(지연)을 느끼고, 서버를 운영하는 쪽은 서버가 1초에 만드는 토큰 수(처리량)를 봅니다. 그래서 두 쪽의 지표를 따로 정의합니다.</p>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/06_metric_overview.png" width="1050" alt="지연(TTFT, TPOT, ITL, 요청 지연)은 사용자 한 명이 기다리는 시간이라 작을수록 좋고, 처리량(출력, 요청)은 서버 전체의 속도라 클수록 좋습니다.">
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">TTFT(Time To First Token, 첫 토큰 지연)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">요청을 보낸 때부터 첫 출력 토큰을 받을 때까지 걸린 시간입니다. 서버에서 기다린 시간, 입력을 처리한 시간과 요청·응답의 통신 시간이 포함됩니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">prefill(입력 처리):</strong> 시스템 메시지와 질문의 토큰을 계산해 KV 캐시를 채우고 첫 토큰을 만드는 단계입니다(지난 단원 교안 01 5-1). 새로 처리할 입력이 길수록 오래 걸립니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시:</strong> 앞 토큰들의 key와 value(K·V) 계산 결과를 GPU 메모리에 저장해 둔 것입니다. 다음 토큰을 만들 때 앞 토큰의 K·V를 다시 계산하지 않고 읽습니다.</li>
</ul>
</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/06_metric_ttft.png" width="760" alt="질문을 보낸 뒤 대기열에서 기다리고 입력을 처리(prefill)해 첫 토큰이 나올 때까지가 TTFT입니다. 입력이 길거나 앞에 기다리는 요청이 많으면 길어집니다.">
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">TPOT(Time Per Output Token, 출력 토큰당 시간)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">첫 토큰 뒤부터 마지막 토큰까지 걸린 시간을 나머지 출력 토큰 수로 나눈 평균입니다. 요청 하나에 값이 하나입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">decode(토큰 생성):</strong> 앞 토큰들의 KV 캐시를 읽으며 다음 토큰을 하나씩 만드는 단계입니다(지난 단원 교안 01 5-1).<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">생성 단계:</strong> 서버가 처리 중인 요청들을 묶어 GPU로 한 번에 계산하는 단위입니다. 일반적인 decode에서는 요청마다 토큰을 하나씩 얻고, 연속 배치는 단계마다 끝난 요청을 빼고 기다리던 요청을 넣습니다(지난 단원 교안 01 5-4).</li>
</ul>
</li>
</ul>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">TPOT</strong> = 첫 토큰 뒤 걸린 시간 ÷ (출력 토큰 수 − 1)</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/06_metric_tpot.png" width="760" alt="첫 토큰 뒤 나머지 토큰 5개를 만든 시간을 5로 나눈 값이 TPOT입니다. 답이 화면에 흘러나오는 속도입니다.">
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">ITL(Inter-Token Latency, 토큰 간 지연)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 벤치마크에서는 연속한 스트리밍 응답(청크)의 도착 간격을 ITL로 기록합니다.</strong> 받는 쪽은 청크가 도착한 시각만 알 수 있기 때문입니다. 평균과 최댓값으로 응답이 고르게 이어지는지 봅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">청크(chunk):</strong> 한 번에 전달되는 응답 조각입니다. 청크마다 토큰이 하나면 ITL은 토큰 간 지연과 같습니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">ITL이 TPOT보다 길어지는 경우:</strong> 한 청크에 토큰이 여러 개 담기면 그 간격에 여러 토큰을 만든 시간이 들어갑니다. 예: 토큰이 20ms마다 만들어지고 한 청크에 3개가 담겨 오면, TPOT는 20ms지만 그 청크의 ITL은 60ms입니다.</li>
</ul>
</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/06_metric_itl_chunks.png" width="950" alt="응답 청크 다섯 개가 0, 20, 40, 100, 120ms에 도착한 예입니다. 4번 청크에는 토큰 3개가 담겨 있습니다. 청크 간격은 20, 20, 60, 20ms라 ITL 평균은 30ms, 최댓값은 60ms이고, 토큰은 20ms마다 만들어져 TPOT는 20ms입니다.">
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요청 지연(end-to-end latency)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">요청을 보낸 때부터 마지막 토큰을 받을 때까지 걸린 시간입니다. 첫 토큰을 기다린 시간과 나머지 토큰을 받을 때까지의 시간을 더한 값입니다.</li>
</ul>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">요청 지연</strong> = TTFT + (출력 토큰 수 − 1) × TPOT</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/06_metric_e2e.png" width="760" alt="질문을 보낸 때부터 마지막 토큰까지가 요청 지연입니다. TTFT에 토큰 생성 시간을 더한 값이라 답이 길수록 길어집니다.">

<p style="margin:6px 0;">TTFT 300ms, 출력 토큰 101개, TPOT 20ms인 요청 하나를 따라가 봅니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">보낸 뒤 300ms에 첫 토큰이 도착합니다.</li>
<li style="margin:3px 0;">나머지 토큰 100개가 20ms 간격으로 2,000ms 동안 이어 도착합니다.</li>
<li style="margin:3px 0;">요청 지연은 300 + 100 × 20 = 2,300ms입니다.</li>
</ol>
<p style="margin:6px 0;">vLLM 벤치마크는 TPOT 식의 '첫 토큰 뒤 걸린 시간'을 요청 지연에서 TTFT를 뺀 값으로 계산합니다(출력 토큰이 2개 이상인 요청만).</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">처리량(throughput)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">출력 처리량(output throughput):</strong> 서버가 1초에 만든 출력 토큰 수(토큰/초)입니다. 요청 하나가 아니라 서버 전체의 속도입니다. 이 교안에서 처리량이라고만 쓰면 출력 처리량입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요청 처리량(request throughput):</strong> 서버가 1초에 끝낸 요청 수(요청/초)입니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요청 1개 완료:</strong> 질문 하나의 응답을 마지막 토큰까지 성공적으로 받은 것입니다. 토큰 하나를 만든 횟수와 구분합니다.</li>
</ul>
</li>
</ul>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">출력 처리량</strong> = 전체 출력 토큰 수 ÷ 전체 측정 시간(초)<br><strong style="color:#1f3a5f;">요청 처리량</strong> = 완료한 요청 수 ÷ 전체 측정 시간(초)</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">두 식 모두 여러 요청을 보내기 시작해 모두 끝날 때까지의 시간을 분모로 씁니다. 10초 동안 질문 20개의 응답을 끝내고 출력 토큰을 총 2,000개 만들었다면, 요청 처리량은 2요청/초이고 출력 처리량은 200토큰/초입니다.</p>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/06_metric_throughput.png" width="760" alt="GPU 서버가 여러 요청의 토큰을 함께 만들 때, 1초 동안 만든 전체 출력 토큰 수가 출력 처리량입니다. 서버 전체의 속도입니다.">
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">지표 정리</strong></p>
<p style="margin:6px 0;">요청을 많이 함께 처리하면 출력 처리량은 커지지만 요청 하나의 지연이 길어질 수 있어, 두 쪽을 함께 봅니다. 아래 시간축은 청크 하나에 토큰 하나가 담겨 도착하는 예입니다.</p>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/01_serving_metrics.png" width="1050" alt="한 청크에 토큰 하나가 담겨 도착하는 예에서 TTFT, ITL, TPOT, 요청 지연의 구간을 비교합니다. 여러 토큰이 한 청크에 담기면 ITL은 청크 사이 간격입니다.">
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">백분위(percentile)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">요청들의 지연을 빠른 순으로 줄 세웠을 때 그 비율 지점의 값입니다. p50은 가운데 값(중앙값), p95는 95% 지점의 값입니다.</li>
<li style="margin:3px 0;">p95는 약 95%의 요청이 이 시간 이하였다는 뜻입니다. 나머지 느린 요청까지 보려면 p99도 함께 봅니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/02_percentiles_exact.png" width="1050" alt="요청 100개를 빠른 순으로 정렬한 산점도입니다. 90개는 200ms, 8개는 900ms, 2개는 3,000ms여서 평균은 312ms, p50은 200ms, p95는 900ms, p99는 3,000ms입니다.">

<p style="margin:6px 0;">요청 100개 가운데 90개의 TTFT가 200ms, 8개가 900ms, 2개가 3,000ms인 경우를 계산해 봅니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">평균:</strong> (90 × 200 + 8 × 900 + 2 × 3,000) / 100 = 312ms입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">p50:</strong> 빠른 순으로 50번째 요청의 200ms입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">p95:</strong> 95번째 요청의 900ms입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">p99:</strong> 99번째 요청의 3,000ms입니다.</li>
</ol>
<p style="margin:6px 0;">평균 312ms만 보면 3초를 기다린 요청 2개가 보이지 않습니다. 그래서 지연 목표는 'p95 TTFT 1초 이하'처럼 백분위로 정합니다. 이 예는 p95로는 목표를 지키고 p99로는 넘깁니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">주로 백분위로 보는 지표</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">지표</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">주로 보는 백분위</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">보는 이유</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">TTFT</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">p95, p99</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">답이 시작되기까지 오래 기다린 사용자가 얼마나 되는지 봅니다. 채팅 서비스의 지연 목표로 흔히 씁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">TPOT</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">p95, p99</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">답이 느리게 이어지는 요청을 찾습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">ITL</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">p99</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">응답 도중 길게 멈춘 순간을 찾습니다. 평균에는 거의 드러나지 않습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">요청 지연</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">p95, p99</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">답 전체를 받아야 쓸 수 있을 때(스트리밍하지 않는 API, 일괄 처리) 봅니다.</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">p50:</strong> 보통 요청이 겪는 값입니다. p95, p99와 나란히 놓고 느린 요청이 보통보다 얼마나 느린지 비교합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">처리량:</strong> 출력 처리량과 요청 처리량은 요청마다 생기는 값이 아니라 측정 전체에서 나오는 값 하나라, 백분위를 보지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">이 교안의 지연 목표:</strong> 5-3의 goodput은 TTFT와 TPOT 두 지표로 정합니다.</li>
</ul>
</div>

## 3. 요청 하나의 지표

### 3-1. 스트리밍 응답 받기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">지난 단원에서 쓴 LangChain <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatOpenAI</code>로 같은 파드의 서버에 요청하면, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">stream</code>이 답을 <strong style="font-size:1.13em;color:#1f3a5f;">청크(응답 조각) 단위</strong>로 돌려줍니다. 청크 하나에는 토큰이 하나 또는 여러 개 들어갈 수 있습니다. 내용이 든 청크가 도착할 때마다 시각을 기록해 TTFT, TPOT, ITL, 요청 지연을 계산합니다.</p>
</div>

**셀 실행 · RunPod 원격 노트북** (VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다)

In [ ]:
from langchain_openai import ChatOpenAI

# 같은 파드의 서버 주소와 API 키로 요청하는 LangChain 채팅 모델(지난 단원과 같은 모양)
llm = ChatOpenAI(
    model=SERVED_NAME,                 # 서버 이름(--served-model-name)
    base_url=f"{SERVER_URL}/v1",       # 같은 파드의 로컬 API 주소 + /v1
    api_key=VLLM_API_KEY,              # 서버를 띄울 때 정한 --api-key
    temperature=0,
    max_tokens=MAX_TOKENS,
    stream_usage=True,                 # base_url을 직접 정하면 기본값이 꺼짐이라, 스트리밍 응답에서도 토큰 사용량을 받으려고 켭니다
)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">시각은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">time.perf_counter()</code>(초 단위, 짧은 구간의 시간을 기록할 때 쓰는 시계)로 기록하고, 받은 글자는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">print(..., end=&quot;&quot;, flush=True)</code>로 도착하는 대로 화면에 씁니다. 다른 질문을 보내 보려면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">user_question</code>의 문자열만 바꿔 다시 실행합니다.</p>
</div>

In [ ]:
import time

# 보낼 질문(노무 질문의 첫 번째)
user_question = "저희 회사에서 쉬는 시간을 빼고 일주일에 40시간이 넘게 일하라고 하는데, 이렇게 근무해도 되나요?"

# 참고 조문이 든 시스템 메시지와 질문(지난 단원과 같은 튜플 리스트)
messages = [("system", system_prompt), ("human", user_question)]

# token_times에는 내용이 든 청크의 도착 시각을 기록하고, 답은 도착하는 대로 화면에 씁니다
# 토큰 사용량(usage_metadata)은 내용이 빈 마지막 청크에 붙어 옵니다
token_times = []
print("질문:", user_question)
print("답: ", end="")
sent_at = time.perf_counter()
for chunk in llm.stream(messages):
    if chunk.content:
        token_times.append(time.perf_counter())
        print(chunk.content, end="", flush=True)
    if chunk.usage_metadata:
        usage = chunk.usage_metadata

In [ ]:
# 내용이 든 청크 수와 토큰 사용량
stream_info = {
    "내용이 든 청크 수": len(token_times),
    "입력 토큰 수(input_tokens)": usage["input_tokens"],
    "출력 토큰 수(output_tokens)": usage["output_tokens"],
}
display(stream_info)

### 3-2. TTFT, TPOT, ITL, 요청 지연 계산

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">기록한 시각으로 네 지표를 계산합니다. TPOT는 vLLM 벤치마크와 같은 식으로, 출력 토큰 수는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage[&quot;output_tokens&quot;]</code>를 씁니다.</p>
</div>

In [ ]:
# TTFT: 보낸 때부터 첫 청크까지(초를 ms로 바꿉니다)
ttft_ms = (token_times[0] - sent_at) * 1000

# 요청 지연: 보낸 때부터 마지막 청크까지
e2e_ms = (token_times[-1] - sent_at) * 1000

# TPOT = (요청 지연 - TTFT) / (출력 토큰 수 - 1)
tpot_ms = (e2e_ms - ttft_ms) / (usage["output_tokens"] - 1)

# ITL: 이웃한 두 청크의 도착 시각 간격 리스트
itl_ms = [(later - earlier) * 1000 for earlier, later in zip(token_times[:-1], token_times[1:])]

# 네 지표를 ms 단위로 한 딕셔너리에 모읍니다. ITL은 간격 리스트의 평균과 최댓값입니다
request_metrics = {
    "TTFT(ms)": round(ttft_ms, 1),
    "TPOT(ms)": round(tpot_ms, 1),
    "ITL 평균(ms)": round(sum(itl_ms) / len(itl_ms), 1),
    "ITL 최댓값(ms)": round(max(itl_ms), 1),
    "요청 지연(ms)": round(e2e_ms, 1),
}
display(request_metrics)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">청크 수와 출력 토큰 수:</strong> 두 수가 같으면 청크 하나에 토큰 하나가 실린 것이라 TPOT와 ITL 평균이 거의 같습니다. 생성 상한 전에 답이 끝나면 내용 없는 대화 끝 토큰 때문에 청크 수가 출력 토큰 수보다 하나 적습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">TTFT와 요청 지연:</strong> TTFT에는 서버에서 기다린 시간과 입력을 prefill한 시간이 들어 있고, 요청 지연의 나머지는 첫 토큰 뒤의 응답을 받을 때까지입니다. 원격 노트북에서 같은 파드의 API 응답 도착 시각으로 측정한 값입니다.</li>
</ul>
</div>

## 4. vllm bench serve로 부하 측정

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">요청 하나의 지표로는 여러 사용자가 동시에 쓸 때의 지연과 처리량을 알 수 없습니다. 그래서 많은 요청을 한꺼번에 보내는 부하 측정을 합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">vllm bench serve</code></strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">vLLM 0.27.0에 들어 있는 서빙 벤치마크 도구입니다. 정한 수의 요청을 정한 동시 요청 수로 서버에 보내고, 요청마다 TTFT, TPOT, ITL, 요청 지연을 계산해 평균, 백분위, 출력 처리량과 요청 처리량을 결과 JSON에 저장합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 요청 수(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--max-concurrency</code>):</strong> 벤치마크 도구가 답을 아직 받지 못한 요청을 최대 몇 개까지 두는지 정하는 값입니다. 요청 하나가 끝나면 다음 요청을 보냅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--request-rate</code>:</strong> 1초에 보낼 요청 수입니다. 이 교안은 주지 않으므로, 도구는 요청 사이에 기다리지 않고 동시 요청 수가 찰 때까지 곧바로 보냅니다.</li>
</ul>
</div>

### 4-1. 벤치마크 입력 파일

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">chat template:</strong> 시스템 메시지와 user 메시지를 모델이 학습한 표시(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;|im_start|&gt;</code> 같은)로 감싸 글 하나로 잇는 규칙입니다.</p>
<p style="margin:6px 0;">벤치마크 도구의 custom 데이터셋은 한 줄에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;prompt&quot;: ...}</code> 하나가 든 JSONL 파일을 읽습니다. 이 도구는 chat template을 user 메시지에만 씌우므로 시스템 메시지를 넣을 수 없습니다. 그래서 원격 노트북에서 시스템 메시지까지 chat template을 씌운 글을 저장하고, 벤치마크에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--skip-chat-template</code>을 줍니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">build_prompts</code>는 과제에서도 쓰므로 함수로 둡니다.</p>
</div>

In [ ]:
def build_prompts(questions, system_prompt, path):
    """질문마다 시스템 메시지와 함께 chat template을 적용한 프롬프트를 vllm bench serve 입력 파일(JSONL)로 저장합니다."""
    with open(path, "w", encoding="utf-8") as f:
        for row in questions:
            messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": row["question"]}]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            f.write(json.dumps({"prompt": prompt}, ensure_ascii=False) + "\n")  # 벤치마크 도구가 읽는 키는 prompt
    print("입력 파일:", path.name, "| 요청 수:", len(questions))

In [ ]:
# 노무 질문 전체를 같은 시스템 메시지와 함께 입력 파일로 저장합니다
labor_prompts_path = output_dir / "labor_prompts.jsonl"
build_prompts(labor_questions, system_prompt, labor_prompts_path)

### 4-2. RunPod에서 벤치마크 실행

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">원격 노트북에서 만든 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/labor_prompts.jsonl</code>을 같은 파드의 벤치마크가 그대로 읽고, 서버(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">127.0.0.1:8000</code>)에 요청합니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm bench serve</code> 명령을 감싼 bash 함수입니다. RunPod 터미널 2에서 아래 블록을 한 번 실행해 함수를 정의합니다. 실습 서버는 API 키를 요구하므로, 벤치마크 도구는 준비 블록에서 넣어 둔 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">OPENAI_API_KEY</code>를 요청에 붙입니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 2**

```bash
# 사용법: run_bench <결과 이름> <입력 파일> <동시 요청 수> <요청 수>
# 같은 파드의 서버(127.0.0.1:8000)에 요청을 보내고, 결과 JSON은 output/bench/<결과 이름>.json에,
# 화면의 요약 테이블은 output/bench/<결과 이름>.txt에도 남깁니다
run_bench() {
  uv run vllm bench serve \
    --backend openai \
    --base-url http://127.0.0.1:8000 \
    --model qwen3.5-4b \
    --tokenizer Qwen/Qwen3.5-4B \
    --dataset-name custom \
    --dataset-path "$2" \
    --skip-chat-template \
    --custom-output-len 256 \
    --num-prompts "$4" \
    --max-concurrency "$3" \
    --temperature 0 \
    --seed 0 \
    --percentile-metrics ttft,tpot,itl,e2el \
    --metric-percentiles 50,95,99 \
    --goodput ttft:1000 tpot:50 \
    --save-result --save-detailed \
    --result-dir output/bench \
    --result-filename "$1.json" | tee "output/bench/$1.txt"
}
```

터미널 2를 새로 열면 준비 블록과 이 블록을 다시 실행합니다.

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">run_bench</code> 옵션</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">옵션</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--backend openai</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">채팅 형식이 아니라 글을 이어 쓰는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/completions</code> 주소로 요청을 보냅니다. 입력 파일에 chat template을 미리 씌웠기 때문입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--base-url http://127.0.0.1:8000</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">같은 파드에서 실행 중인 서버 주소입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--model qwen3.5-4b</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">요청의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code>에 넣는 서버 이름입니다. API 키가 걸린 서버에는 꼭 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--tokenizer Qwen/Qwen3.5-4B</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">요청을 만들 때 입력 토큰 수를 세는 토크나이저입니다. 서버가 응답에 토큰 사용량을 주면 결과 JSON의 토큰 수는 서버 값을 씁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--dataset-name custom</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--dataset-path</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;prompt&quot;: ...}</code> 한 줄씩 든 입력 파일(함수의 둘째 인자)을 읽습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--skip-chat-template</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">입력 파일에 미리 적용한 chat template을 다시 씌우지 않습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--custom-output-len 256</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">요청마다 생성 상한입니다(지난 단원과 같은 256토큰).</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--num-prompts</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-concurrency</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">보낼 요청 수(넷째 인자)와 동시 요청 수(셋째 인자)입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--temperature 0</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--seed 0</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">확률이 가장 큰 토큰을 고르는 그리디 생성과 입력 파일의 요청을 섞는 순서를 정합니다. 배치가 달라지면 같은 질문의 답도 조금 달라질 수 있습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--percentile-metrics ttft,tpot,itl,e2el</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">백분위를 계산할 지표입니다. e2el이 요청 지연입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--metric-percentiles 50,95,99</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">결과 JSON에 p50, p95, p99 값을 남깁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--goodput ttft:1000 tpot:50</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">지연 목표(TTFT 1000ms, TPOT 50ms)입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--save-result --save-detailed</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">결과 JSON을 저장하고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ttfts</code>(요청별 첫 토큰 지연 목록, 초 단위) 같은 개별 요청의 측정값도 넣습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--result-dir</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--result-filename</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">결과 JSON을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/bench/&lt;결과 이름&gt;.json</code>에 저장합니다. 화면의 요약 테이블은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code>로 같은 이름의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.txt</code>에도 남깁니다.</td>
</tr>
</tbody>
</table>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">동시 요청 16으로 노무 질문 64개를 보냅니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 2**

```bash
# 동시 요청 16으로 요청 64개를 보냅니다(--seed 0으로 섞은 순서의 앞 64개라, 설정을 바꿔도 같은 요청을 보냅니다)
run_bench labor_load_c16 output/labor_prompts.jsonl 16 64
```

### 4-3. RunPod 화면에서 요약 테이블 읽기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code>는 화면의 요약 테이블을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/bench/labor_load_c16.txt</code>에도 남기므로, 터미널 2에서 다시 열어 봅니다. TTFT, TPOT, ITL, E2EL(요청 지연)마다 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Mean</code>(평균), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Median</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">P50</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">P95</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">P99</code> 줄이 있고 값은 ms입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Median</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">P50</code>과 같은 중앙값입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Output token throughput</code>이 출력 처리량, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Request throughput</code>이 요청 처리량입니다. 바로 옆 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Total token throughput</code>은 입력 토큰까지 더한 값이라 출력 처리량이 아닙니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Peak concurrent requests</code>는 정한 동시 요청 수보다 클 수 있으니, 동시 요청 수는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Maximum request concurrency</code>로 확인합니다.</p>
</div>

**RunPod 터미널 2 · 화면에서 확인**

```bash
cat output/bench/labor_load_c16.txt
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Serving Benchmark Result</code> 아래 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Successful requests</code>가 64</strong>인지 먼저 확인합니다. 64보다 작으면 RunPod 터미널 1의 서버 화면에 오류가 있는지 봅니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">평균과 p50:</strong> TTFT처럼 늦은 요청이 몇 개 있는 지표는 평균이 p50보다 커집니다. 긴 입력은 여러 생성 단계에 나눠 prefill 하므로, 한꺼번에 들어온 요청은 차례로 기다리고 TTFT에서 이 차이가 커지기 쉽습니다. 평균은 대부분의 요청과 느린 꼬리 어느 쪽도 대표하지 않으므로 지연 목표는 p95로 판단합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">p95 TTFT와 p95 TPOT:</strong> 각각 지연 목표(TTFT 1000ms, TPOT 50ms) 안에 드는지 봅니다. 두 목표를 한 요청이 함께 지켰는지는 5-3의 목표 달성률로 판정합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">반복 요청의 TTFT:</strong> 같은 시스템 메시지를 이미 처리했다면 vLLM이 그 계산 결과를 다시 써서(접두어 캐싱) 첫 토큰이 빨라질 수 있습니다. 입력 길이와 동시 요청 수도 함께 달라지므로 요청 하나의 TTFT와 같은 값이라고 보지는 않습니다.</li>
</ul>
</div>

## 5. 동시 요청 수와 지연, 처리량

### 5-1. 동시 요청 수 바꾸기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 입력 파일로 동시 요청 수를 1, 16, 64로 바꿔 벤치마크합니다. 측정 시간을 줄이기 위해 각각 요청 8, 64, 64개를 보내며, 동시 요청 16은 앞에서 측정한 결과를 그대로 씁니다. 새로 돌리는 측정은 두 번이고, 요청 수가 적으므로 결과는 장시간 운영 성능보다 설정의 차이를 살피는 데 씁니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">벤치마크를 돌리며 서버 로그 보기:</strong> 서버는 약 10초마다 처리량 한 줄을 로그에 남깁니다. 원격 저장소 폴더에서 새 터미널(RunPod 터미널 3)을 열고 아래 블록을 켜 둡니다. 이 터미널은 API 키가 필요 없어 단원 폴더로만 이동합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 3**

```bash
# 원격 실습 저장소 폴더에서 연 터미널에서 실행합니다
cd day76_추론가속_서빙벤치마킹
pwd   # 끝이 day76_추론가속_서빙벤치마킹이면 맞습니다

# 서버 로그에 새로 찍히는 처리량 줄만 바로바로 보여 줍니다(멈출 때까지 계속 돕니다)
tail -f output/qwen3.5-4b_server.log | grep --line-buffered "throughput"
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">명령어 설정</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tail -f output/qwen3.5-4b_server.log</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">로그 파일 끝에 새로 붙는 줄을 계속 보여 줍니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">grep --line-buffered &quot;throughput&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">그 가운데 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">throughput</code>이 든 줄만, 모아 두지 않고 한 줄씩 바로 내보냅니다.</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">터미널 3을 켜 둔 채, 터미널 2에서 아래 두 줄을 실행합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널 2**

```bash
# 동시 요청 수와 보낼 요청 수만 바꿔 두 번 실행합니다(동시 요청 16은 4-2의 결과를 그대로 씁니다)
# 동시 요청 1은 요청을 하나씩 처리해 오래 걸리므로 8개만 보내고, 동시 요청 64는 64개를 보냅니다
run_bench labor_load_c1 output/labor_prompts.jsonl 1 8
run_bench labor_load_c64 output/labor_prompts.jsonl 64 64
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">터미널 3 화면 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">입력 처리량(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Avg prompt throughput</code>):</strong> 로그에 집계된 새 입력 토큰 수를 앞 줄 이후 구간(약 10초)의 길이로 나눈 값입니다. 접두어 캐시에서 다시 쓴 토큰은 빠지고, 분모는 prefill에만 쓴 시간이 아니므로 prefill 자체의 속도는 아닙니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">출력 처리량(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Avg generation throughput</code>):</strong> 같은 구간에 모든 요청이 받은 출력 토큰(prefill 끝에 나온 첫 토큰 포함)을 더해 구간 길이로 나눈 서버 전체 값입니다. 동시 요청이 늘면 이 값이 커지면서 요청 하나의 TTFT와 TPOT는 길어질 수 있습니다. 요청 하나의 속도는 아래 식으로 봅니다.</li>
</ul>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">요청 하나의 생성 속도</strong> = 1,000 ÷ TPOT(ms/토큰)</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Running</code>:</strong> 줄을 찍는 그 순간 처리 중인 요청 수입니다. 그래서 벤치마크가 끝나 가는 줄은 출력 처리량이 커도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Running</code>이 작거나 0일 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Prefix cache hit rate</code>:</strong> 입력 토큰 가운데 접두어 캐시에서 찾은 비율입니다. 최근 요청들의 누적 비율이라, 요청이 없는 줄에도 높은 값이 그대로 남습니다. 벤치마크 중에 크게 올라 있으면 접두어 캐싱이 동작하는 것이고, 0.0%에 머물면 서버 명령의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code>을 확인합니다.</li>
</ul>
<p style="margin:6px 0;">두 벤치마크가 끝나면 터미널 3에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 멈춥니다. 로그 항목을 자세히 읽는 방법은 부록 01에 있습니다.</p>
<p style="margin:6px 0;">동시 요청 16의 앞 측정까지 결과 세 개가 같은 파드의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/bench/</code>에 저장되어 있습니다. 원격 노트북에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_load_c1.json</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_load_c16.json</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor_load_c64.json</code>을 바로 읽어 비교합니다.</p>
</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">결과 JSON의 지표 키는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;통계&gt;_&lt;지표&gt;_ms</code> 모양입니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">mean_ttft_ms</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">p95_ttft_ms</code>). 지표 이름은 ttft, tpot, itl, e2el(요청 지연)이고, 처리량은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_throughput</code>(출력 처리량), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request_throughput</code>(요청 처리량)입니다.</p>
</div>

**셀 실행 · RunPod 원격 노트북** (VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다)

In [ ]:
# 동시 요청 수(작은 순서). 동시 요청 16은 4-2에서 실행한 labor_load_c16.json입니다
concurrency_levels = [1, 16, 64]

# 동시 요청 수마다 같은 파드에 저장된 결과 JSON을 딕셔너리로 읽습니다
load_results = {}
for concurrency in concurrency_levels:
    result_path = bench_dir / f"labor_load_c{concurrency}.json"
    load_results[concurrency] = json.loads(result_path.read_text(encoding="utf-8"))
    print(f"동시 요청 {concurrency}: 완료 {load_results[concurrency]['completed']}개 / 보낸 요청 {load_results[concurrency]['num_prompts']}개")

In [ ]:
# 동시 요청 수마다 출력 처리량, 요청 처리량, p95 지연을 한 행으로 모읍니다
load_rows = []
for concurrency in concurrency_levels:
    result = load_results[concurrency]
    load_rows.append({
        "동시 요청 수": concurrency,
        "출력 처리량(토큰/초)": result["output_throughput"],
        "요청 처리량(요청/초)": result["request_throughput"],
        "p95 TTFT(ms)": result["p95_ttft_ms"],
        "p95 TPOT(ms)": result["p95_tpot_ms"],
    })

# 요청 처리량이 1보다 작을 수 있어 소수 둘째 자리까지 봅니다
load_table = pd.DataFrame(load_rows).round(2)
display(load_table)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">완료 요청 수:</strong> 줄마다 완료 요청 수가 보낸 요청 수와 같아야 그 행의 지표를 믿을 수 있습니다.</li>
</ul>
</div>

### 5-2. 처리량과 지연 그래프

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">위 테이블을 두 그래프로 그립니다. 동시 요청 수가 1, 16, 64로 크게 벌어지므로 가로축은 로그(밑 2)로 둡니다.</p>
</div>

In [ ]:
# 그래프 두 개를 나란히 둡니다. 가로축 눈금은 측정한 동시 요청 수입니다
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
tick_labels = [str(level) for level in concurrency_levels]

# 왼쪽: 동시 요청 수와 출력 처리량
axes[0].plot(load_table["동시 요청 수"], load_table["출력 처리량(토큰/초)"], marker="o", color="tab:green", label="출력 처리량")
axes[0].set_xscale("log", base=2)
axes[0].set_xticks(concurrency_levels, labels=tick_labels)
axes[0].set_title("동시 요청 수와 출력 처리량")
axes[0].set_xlabel("동시 요청 수(개)")
axes[0].set_ylabel("출력 처리량(토큰/초)")
axes[0].legend()

# 오른쪽: 동시 요청 수와 p95 TTFT, p95 TPOT. 점선은 지연 목표입니다
axes[1].plot(load_table["동시 요청 수"], load_table["p95 TTFT(ms)"], marker="o", color="tab:blue", label="p95 TTFT")
axes[1].plot(load_table["동시 요청 수"], load_table["p95 TPOT(ms)"], marker="s", color="tab:orange", label="p95 TPOT")
axes[1].axhline(TTFT_TARGET_MS, color="tab:blue", linestyle="--", label=f"TTFT 지연 목표 {TTFT_TARGET_MS}ms")
axes[1].axhline(TPOT_TARGET_MS, color="tab:orange", linestyle="--", label=f"TPOT 지연 목표 {TPOT_TARGET_MS}ms")
axes[1].set_xscale("log", base=2)
axes[1].set_yscale("log")  # TTFT와 TPOT는 크기가 크게 달라 세로축도 로그로 둡니다
axes[1].set_xticks(concurrency_levels, labels=tick_labels)
axes[1].set_title("동시 요청 수와 p95 지연")
axes[1].set_xlabel("동시 요청 수(개)")
axes[1].set_ylabel("지연(ms, 로그 눈금)")
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">출력 처리량:</strong> 동시 요청을 늘리면 여러 요청을 함께 계산해 GPU를 더 활용할 수 있습니다. 16에서 64로 요청 수를 4배 늘려도 출력 처리량이 조금만 늘면, 이 부하에서는 추가 요청을 함께 처리하는 이득이 작다는 뜻입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">p95 지연:</strong> 함께 계산하는 요청이 많아지면 단계 하나가 길어지고, 새 요청의 prefill도 다른 요청의 decode와 GPU를 나눠 써 p95 지연이 늘 수 있습니다. 각 선이 지연 목표 점선을 넘는 동시 요청 수를 찾습니다. 점이 점선 위에 겹쳐 보이면 테이블 값으로 판단합니다.</li>
</ul>
</div>

### 5-3. 지연 목표와 goodput

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">goodput</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">지연 목표를 모두 지킨 요청만 센 요청 처리량(요청/초)입니다. vLLM 벤치마크는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--goodput ttft:1000 tpot:50</code>로 받은 목표를 요청마다 모두 지켰는지 보고, 지킨 요청 수를 걸린 시간으로 나눕니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 처리량이 커도 목표를 넘긴 요청은 사용자에게 느린 답이므로, 쓸모 있는 처리량만 셉니다.</li>
</ul>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">goodput</strong> = 지연 목표를 모두 지킨 요청 수 ÷ 걸린 시간(초)</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목표 달성률(attainment)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">goodput을 요청 처리량으로 나눈 값으로, 전체 요청 가운데 지연 목표를 모두 지킨 요청의 비율입니다.</li>
<li style="margin:3px 0;">지연 목표가 하나면 'p95가 목표 안'과 목표 달성률 0.95 이상은 거의 같은 기준입니다. 이 교안처럼 TTFT와 TPOT를 함께 보면 두 목표를 모두 지킨 요청만 세므로 목표 달성률이 더 엄격합니다.</li>
</ul>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/03_goodput.png" width="1050" alt="요청 64개를 20초에 끝내고 그 가운데 48개가 두 지연 목표를 모두 지킨 예입니다. 요청 처리량은 3.2요청/초, goodput은 2.4요청/초, 목표 달성률은 0.75입니다.">

<p style="margin:6px 0;">요청 64개를 20초에 끝냈고 그 가운데 48개가 두 목표를 모두 지켰다면, 요청 처리량은 64 / 20 = 3.2요청/초, goodput은 48 / 20 = 2.4요청/초, 목표 달성률은 48 / 64 = 0.75입니다. TPOT 50ms 이하라는 목표는 첫 토큰 뒤에 평균 초당 20토큰 이상을 받는다는 뜻입니다.</p>
<p style="margin:6px 0;">5-1의 테이블에 goodput과 목표 달성률 열을 더합니다.</p>
</div>

In [ ]:
# 동시 요청 수마다 goodput과 목표 달성률(두 지연 목표를 모두 지킨 요청의 비율)을 더합니다
for row in load_rows:
    result = load_results[row["동시 요청 수"]]
    row["goodput(요청/초)"] = result["request_goodput"]
    row["목표 달성률"] = result["request_goodput"] / result["request_throughput"]  # 반올림 전 값으로 나눕니다

load_table = pd.DataFrame(load_rows).round(2)
display(load_table)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">고르는 기준</strong> (아래 코드는 이 기준을 그대로 옮긴 것입니다)</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">지연 목표를 지킨 동시 요청 수만 후보로 남깁니다:</strong> 목표 달성률(TTFT 1,000ms와 TPOT 50ms를 모두 지킨 요청의 비율)이 <strong style="font-size:1.13em;color:#1f3a5f;">0.95 이상</strong>인 동시 요청 수만 후보입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">후보 가운데 출력 처리량이 가장 큰 값을 고릅니다:</strong> 서버 한 대가 토큰을 가장 많이 만드는 값이라 출력 토큰당 비용이 가장 쌉니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">후보가 없으면 고르지 않습니다:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">best_concurrency</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">None</code>으로 남습니다.</li>
</ol>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">기준</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">코드의 조건</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">1. 목표 달성률 0.95 이상</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">row[&quot;목표 달성률&quot;] &gt;= 0.95</code></td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">2. 출력 처리량이 가장 큰 값</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">row[&quot;출력 처리량(토큰/초)&quot;] &gt; best_throughput</code>이면 그 값으로 바꿔 둡니다</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">3. 후보가 없을 때</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">반복이 끝나도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">best_concurrency is None</code></td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">비교는 반올림 전 값인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">load_rows</code>로 합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">load_table</code>은 둘째 자리에서 반올림한 값이라 0.946도 0.95로 보이기 때문입니다.</p>
</div>

In [ ]:
# 목표 달성률이 0.95 이상인 동시 요청 수 가운데 출력 처리량이 가장 큰 값을 고릅니다
# 반올림 전 값인 load_rows에서 고릅니다(테이블에서는 0.946도 0.95로 보입니다)
best_concurrency = None
best_throughput = 0
for row in load_rows:
    if row["목표 달성률"] >= 0.95 and row["출력 처리량(토큰/초)"] > best_throughput:
        best_concurrency = row["동시 요청 수"]
        best_throughput = row["출력 처리량(토큰/초)"]
print(f"지연 목표 TTFT {TTFT_TARGET_MS}ms, TPOT {TPOT_TARGET_MS}ms를 95% 이상 지키는 동시 요청 수: {best_concurrency} | 출력 처리량: {best_throughput:.1f} 토큰/초")

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">goodput과 목표 달성률:</strong> goodput은 동시 요청을 늘리면 처음에는 커지다가, 목표를 넘기는 요청이 많아지면 줄어 0이 되기도 합니다. goodput이 가장 큰 값도 목표 달성률이 0.95보다 낮을 수 있어, goodput 하나만 보고 고르면 느린 쪽 5%보다 많은 사용자가 목표보다 오래 기다리는 설정을 고르게 됩니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">best_concurrency</code>:</strong> 지연 목표를 95% 이상 지키면서 이 서버 한 대가 가장 많은 토큰을 만드는 동시 요청 수입니다. 실무에서는 이 값을 서버 한 대에 동시에 보낼 요청 수의 기준으로 쓰고, 사용자가 더 많으면 서버를 늘립니다. GPU가 다르면 고른 값도 달라질 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">값이 바뀌는 경우:</strong> 요청 64개에서는 목표를 넘긴 요청이 3개 이하일 때 목표 달성률 0.95 이상입니다. 이 경계에 가까우면 다시 측정했을 때 선택이 바뀔 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">1 또는 None인 경우:</strong> 1은 측정한 후보 가운데 선택된 값일 뿐, 실제 상한을 알려 주지는 않습니다. 더 촘촘한 동시 요청 수를 측정해야 상한을 좁힐 수 있습니다. None이면 목표 달성률 95% 이상인 후보가 없으므로 지연 목표나 서버 설정을 다시 봅니다.</li>
</ul>
</div>

## 6. 100만 토큰당 비용

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">GPU 요금은 빌린 시간에 붙으므로, 출력 토큰 100만 개를 만드는 데 드는 비용은 시간당 요금을 한 시간에 만드는 토큰 수로 나눠 계산합니다.</p>
</div>

<div class="day76-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">100만 토큰당 비용</strong> = GPU 시간당 요금 ÷ (출력 처리량 × 3600) × 1,000,000</div>

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">시간당 요금 0.57달러인 GPU가 1,000토큰/초를 만들면 한 시간에 360만 토큰을 만들므로, 100만 토큰당 비용은 0.57 / 3.6 = 약 0.16달러입니다.</p>
</div>

In [ ]:
GPU_PRICE_PER_HOUR = 0.57  # 시간당 요금(달러). RunPod 콘솔에 표시된 내 파드의 시간당 요금으로 바꿉니다

# 100만 토큰당 비용 = 시간당 요금 / (출력 처리량 * 3600) * 1_000_000
load_table["100만 토큰당 비용(달러)"] = (GPU_PRICE_PER_HOUR / (load_table["출력 처리량(토큰/초)"] * 3600) * 1_000_000).round(3)
display(load_table[["동시 요청 수", "출력 처리량(토큰/초)", "목표 달성률", "100만 토큰당 비용(달러)"]])

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">비용과 출력 처리량:</strong> 비용은 출력 처리량에 반비례해, 동시 요청 1처럼 요청을 하나씩 처리하면 비용이 커집니다. 동시 요청 수가 가장 큰 행이 가장 쌀 수 있지만, 지연 목표를 지키지 못하면 쓸 수 없습니다. 그래서 쓸 수 있는 가장 싼 비용은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">best_concurrency</code> 행의 값입니다(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">best_concurrency</code>가 None이면 쓸 수 있는 행이 없습니다).</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GPU가 쉬는 시간:</strong> 이 계산은 GPU가 한 시간 내내 그 동시 요청 수로 일한다고 봅니다. 실제 서비스는 요청이 없는 시간에도 요금이 붙어 비용이 이보다 커집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">출력 토큰 기준:</strong> 이 비용은 입력을 처리(prefill)한 GPU 시간까지 모두 출력 토큰 수로만 나눈 값이라, 입력 처리 비용이 출력 토큰 가격 안에 들어 있습니다. 반면 API 요금은 입력 토큰과 출력 토큰에 가격을 따로 매깁니다. 그래서 API와 비교할 때는 같은 요청에 API로 낼 금액(입력 토큰 수 × 입력 단가 + 출력 토큰 수 × 출력 단가)을 출력 토큰 수로 나눠, 출력 토큰 100만 개당 금액으로 바꾼 뒤 비교합니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">예: 요청 하나가 입력 약 5,000토큰, 출력 256토큰이라면 API 금액은 5,000 × 입력 단가 + 256 × 출력 단가이고, 이 금액을 256으로 나누고 100만을 곱한 값을 위 테이블의 100만 토큰당 비용과 비교합니다.</li>
</ul>
</li>
</ul>
</div>

## 7. 🖐️ 함께 따라하기: 전월세 상담 창구 측정하기

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">본실습의 입력 파일 만들기, 벤치마크, 비용 계산을 같은 서버의 전월세 상담 창구에 적용합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">본실습과 다른 점</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">시스템 메시지:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SYSTEM_PROMPT_LEASE</code> 뒤에 주택임대차보호법(계약 기간과 갱신, 차임과 보증금 증감) 조문을 붙입니다. 참고 조문은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/따라하기/lease_reference.json</code>입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">질문:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/따라하기/lease_questions.jsonl</code>(73개), 입력 파일은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/lease_prompts.jsonl</code></li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 요청 수:</strong> 4와 32(보낼 요청 수 16개와 32개), 결과 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_load_c4</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_load_c32</code></li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 진행 방법:</strong> 코드 셀은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>만 주석에 맞춰 채우고, RunPod 터미널 명령은 <strong style="font-size:1.13em;color:#1f3a5f;">작성할 것:</strong>에 적힌 대로 직접 씁니다. 본실습의 변수와 함수(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tokenizer</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">build_prompts</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU_PRICE_PER_HOUR</code>)와 RunPod 터미널 2의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code>를 그대로 쓰므로 본실습을 먼저 실행합니다. 완성 코드는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/교안_01_정답.ipynb</code>의 같은 절에 있습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">서버는 본실습에서 띄운 것을 그대로 씁니다(모델 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Qwen/Qwen3.5-4B</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-model-len 8192</code>).</li>
<li style="margin:3px 0;">벤치마크는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code> 그대로입니다(생성 상한 256토큰, 지연 목표 TTFT 1000ms, TPOT 50ms).</li>
</ul>
</div>

### 7-1. 데이터와 입력 파일

In [ ]:
# 전월세 상담 창구의 지침 문장
SYSTEM_PROMPT_LEASE = "너는 전월세 상담 챗봇이다. 주택임대차보호법 조문에 근거해 존댓말로 간결히 답한다."


# 전월세 참고 조문과 질문
with open(data_dir / "따라하기" / "lease_reference.json", encoding="utf-8") as f:
    lease_reference = json.load(f)
lease_questions = load_jsonl(data_dir / "따라하기" / "lease_questions.jsonl")

# 시스템 메시지 = 지침 문장 + 빈 줄 + [참고 조문] + 조문들(본실습과 같은 모양)
lease_articles = [article["text"] for article in lease_reference["articles"]]
lease_system_prompt = SYSTEM_PROMPT_LEASE + "\n\n[참고 조문]\n" + "\n".join(lease_articles)
print("참고 조문:", lease_reference["title"], "| 조문 수:", len(lease_articles), "| 질문 수:", len(lease_questions))
print("시스템 메시지 토큰 수:", len(tokenizer.encode(lease_system_prompt)))

<details><summary>힌트</summary>

```text
접근방법: 4-1을 참고합니다.
build_prompts에는 질문 리스트, 시스템 메시지, 저장 경로를 이 순서로 넘깁니다.
질문과 시스템 메시지는 바로 위 셀에서 만든 전월세 것을 씁니다.
```

</details>

In [ ]:
# 🖐️ 함께 따라하기: 전월세 질문과 전월세 시스템 메시지로 벤치마크 입력 파일을 만드세요
lease_prompts_path = output_dir / "lease_prompts.jsonl"
build_prompts(..., ..., lease_prompts_path)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준</strong>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">입력 파일: lease_prompts.jsonl | 요청 수: 73</code>이 출력됩니다.</p>
</div>

### 7-2. RunPod에서 동시 요청 수 두 개로 벤치마크

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성할 것:</strong> RunPod 터미널 2에서 실행할 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench</code> 두 줄을 작성합니다. 형식은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_bench &lt;결과 이름&gt; &lt;입력 파일&gt; &lt;동시 요청 수&gt; &lt;요청 수&gt;</code>이고, 결과 이름은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_load_c&lt;동시 요청 수&gt;</code>, 입력 파일은 RunPod 단원 폴더 기준 경로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/lease_prompts.jsonl</code>입니다. 동시 요청 4는 요청 16개, 동시 요청 32는 요청 32개를 보냅니다.</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 4-2와 5-1을 참고합니다.
본실습의 run_bench 한 줄에서 결과 이름과 입력 파일을 전월세 것으로 바꿉니다.
셋째 인자는 동시 요청 수, 넷째 인자는 보낼 요청 수입니다.
```

</details>

**복사해서 실행 · RunPod 터미널 2**

```bash
# 🖐️ 함께 따라하기: 줄마다 <...> 네 자리를 결과 이름, 입력 파일, 동시 요청 수, 요청 수로 바꿔 실행하세요
run_bench <결과 이름> <입력 파일> <동시 요청 수> <요청 수>
run_bench <결과 이름> <입력 파일> <동시 요청 수> <요청 수>
```

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준</strong>: 터미널 2의 두 요약 테이블에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Successful requests</code>가 16, 32이고, RunPod <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/bench</code>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_load_c4.json</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_load_c32.json</code>이 생깁니다.</p>
</div>

### 7-3. 테이블과 100만 토큰당 비용

<details><summary>힌트</summary>

```text
접근방법: 5-1을 참고합니다.
결과 JSON 파일 이름은 결과 이름 뒤에 .json을 붙인 것이고, 저장된 폴더는 bench_dir입니다.
```

</details>

**셀 실행 · RunPod 원격 노트북** (VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다)

In [ ]:
# 🖐️ 함께 따라하기: 같은 파드에 저장된 두 결과 JSON의 경로(bench_dir / "<결과 이름>.json")를 채우세요
lease_concurrency_levels = [4, 32]
lease_results = {
    4: json.loads((...).read_text(encoding="utf-8")),
    32: json.loads((...).read_text(encoding="utf-8")),
}

# 두 결과 JSON의 완료 요청 수와 보낸 요청 수
for concurrency in lease_concurrency_levels:
    print(f"동시 요청 {concurrency}: 완료 {lease_results[concurrency]['completed']}개 / 보낸 요청 {lease_results[concurrency]['num_prompts']}개")

<details><summary>힌트</summary>

```text
접근방법: 6절을 참고합니다.
시간당 요금은 본실습의 GPU_PRICE_PER_HOUR, 출력 처리량은 lease_table의 출력 처리량 열입니다.
```

</details>

In [ ]:
# 동시 요청 수마다 출력 처리량, p95 TTFT, p95 TPOT, goodput을 한 행으로 모읍니다
lease_rows = []
for concurrency in lease_concurrency_levels:
    result = lease_results[concurrency]
    lease_rows.append({
        "동시 요청 수": concurrency,
        "출력 처리량(토큰/초)": result["output_throughput"],
        "p95 TTFT(ms)": result["p95_ttft_ms"],
        "p95 TPOT(ms)": result["p95_tpot_ms"],
        "goodput(요청/초)": result["request_goodput"],
    })
lease_table = pd.DataFrame(lease_rows).round(2)

# 🖐️ 함께 따라하기: 100만 토큰당 비용 = 시간당 요금 / (출력 처리량 * 3600) * 1_000_000 의 두 빈칸을 채우세요
lease_table["100만 토큰당 비용(달러)"] = (... / (... * 3600) * 1_000_000).round(3)
display(lease_table)

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">확인 기준</strong>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_table</code>에 동시 요청 4와 32의 두 행이 있고, 끝 열 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">100만 토큰당 비용(달러)</code>이 채워져 있습니다.</p>
</div>

### 7-4. 서버는 켜 둡니다

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">실습 서버를 끄지 않고 과제 01로 넘어갑니다. 오늘 실습을 여기서 마치면 아래 접힌 블록을 따라 서버를 끕니다.</p>
</div>

<details><summary>오늘 실습을 여기서 마칠 때</summary>

**RunPod 터미널 1**(`qwen3.5-4b` 서버를 띄운 터미널)에서 **`Ctrl+C` 키**를 누릅니다. 명령을 입력할 수 있게 되면 `nvidia-smi`를 실행합니다.

**GPU 메모리 사용량(`Memory-Usage`)이 서버를 띄우기 전 수준으로 줄었으면** 서버가 꺼진 것입니다. 모델 가중치와 KV 캐시가 쓰던 메모리가 이때 반환됩니다. RunPod에서는 `nvidia-smi`의 프로세스 목록이 비어 보일 수 있어 메모리 사용량으로 판단합니다.

</details>

## 8. 핵심 정리

<div class="day76-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실행 위치:</strong> VS Code Remote SSH로 접속한 RunPod에서 모두 실행합니다. 서버·벤치마크·로그는 원격 터미널, API 요청·파일 생성·결과 분석은 원격 노트북이 맡고 같은 단원 폴더의 파일을 씁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">지연 지표:</strong> TTFT는 보낸 때부터 첫 토큰까지, TPOT는 첫 토큰 뒤 토큰 하나에 걸린 평균 시간, vLLM의 ITL은 연속한 응답 청크 사이 도착 간격, 요청 지연은 마지막 토큰까지의 시간입니다. 요청 지연 = TTFT + (출력 토큰 수 - 1) × TPOT입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">처리량:</strong> 요청 처리량은 질문 하나의 응답을 마지막 토큰까지 완료한 요청 수/초이고, 출력 처리량은 전체 출력 토큰 수/초입니다. 둘 다 전체 측정 시간으로 나눕니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요청 하나의 계산:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatOpenAI(..., stream_usage=True)</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">stream</code>으로 답을 받아 내용이 든 청크의 도착 시각을 기록하면 네 지표를 계산합니다. 출력 토큰 수는 마지막 청크의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage_metadata</code>에 든 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_tokens</code>입니다. 같은 파드의 API 응답 도착 시각으로 계산합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">vllm bench serve</code>:</strong> chat template을 미리 씌운 입력 파일을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--dataset-name custom --skip-chat-template</code>으로 정한 동시 요청 수(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-concurrency</code>)만큼 보내고, 화면 요약 테이블과 결과 JSON에 평균, p50, p95, p99 지표와 처리량을 남깁니다. 결과 하나는 터미널 화면에서 읽고, 여러 결과는 같은 파드에 저장된 JSON을 원격 노트북에서 읽어 비교합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">백분위:</strong> 평균은 느린 요청을 가리므로, 지연 목표는 p95, p99로 정합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">동시 요청 수와 goodput:</strong> 동시 요청 수 1, 16, 64를 같은 입력 파일로 비교하고, 지연 목표(TTFT 1000ms, TPOT 50ms)를 지킨 요청의 비율(목표 달성률 = goodput / 요청 처리량)이 0.95 이상인 값 가운데 출력 처리량이 가장 큰 동시 요청 수를 고릅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버 로그의 처리량:</strong> 벤치마크를 돌리며 터미널 3에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tail -f</code>로 서버 전체 처리량을 봅니다. 처리량은 약 10초 구간의 평균이고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Running</code>은 로그를 찍는 순간의 요청 수라, 둘이 함께 움직이지 않을 수도 있습니다. 입력 처리량은 prefill 속도가 아니고, 요청 하나의 생성 속도는 1,000 ÷ TPOT(ms/토큰)입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">100만 토큰당 비용:</strong> GPU 시간당 요금 / (출력 처리량 × 3600) × 10^6입니다. 같은 GPU에서 처리량이 클수록 작아지고, 출력 토큰만 센 값입니다.</li>
</ul>
</div>